In [3]:
import soccerdata as sd
import pandas as pd
import time

# Premier League seasons from 2016-17 through 2024-25 (inclusive)
seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2016, 2025)]

# Initialize the scraper for all selected seasons
fbref = sd.FBref(leagues="ENG-Premier League", seasons=seasons)

# Read schedule to get match ids, then fetch match-by-match so a single failed match
# download doesn't abort the whole multi-season scrape.
schedule = fbref.read_schedule()
schedule_df = schedule.reset_index()

# Find the match id column (name varies by soccerdata version)
id_candidates = [
    c for c in schedule_df.columns
    if any(k in str(c).lower() for k in ["match_id", "game_id", "matchid", "gameid"])
    or str(c).lower() in {"id", "match"}
    or str(c).lower().endswith("_id")
    or "match" in str(c).lower() and "id" in str(c).lower()
    or "game" in str(c).lower() and "id" in str(c).lower()
  ]
if id_candidates:
    match_id_col = id_candidates[0]
    match_ids = (
        schedule_df[match_id_col]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s.ne("")]
        .drop_duplicates()
        .tolist()
    )
else:
    # Fallback: try index level names
    idx_names = [str(n).lower() for n in schedule.index.names]
    idx_match_levels = [
        n for n in schedule.index.names
        if n is not None and any(k in str(n).lower() for k in ["match", "game"]) and "id" in str(n).lower()
    ]
    if not idx_match_levels:
        raise KeyError(
            "Could not identify match id column/level in fbref.read_schedule(). "
            f"Columns (first 30): {list(schedule_df.columns)[:30]}; index names: {schedule.index.names}"
        )
    level_name = idx_match_levels[0]
    match_ids = (
        pd.Index(schedule.index.get_level_values(level_name))
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

print(f"Seasons: {seasons[0]} → {seasons[-1]}")
print(f"Matches found in schedule: {len(match_ids):,}")

dfs = []
failed = []
for i, match_id in enumerate(match_ids, start=1):
    try:
        df = fbref.read_player_match_stats(
            stat_type="defense",
            match_id=match_id,
            force_cache=True,
        )
        if df is not None and not df.empty:
            dfs.append(df)
    except Exception as e:
        failed.append((match_id, type(e).__name__, str(e)))
    
    # Light progress logging + gentle pacing (helps reduce transient blocking)
    if i % 200 == 0:
        print(f"Fetched {i:,}/{len(match_ids):,} matches... (failures so far: {len(failed)})")
        time.sleep(0.5)

if not dfs:
    raise RuntimeError(
        "No match stats were fetched. If FBref is blocking requests, try rerunning later "
        "or running from a different network."
    )

player_stats_all = pd.concat(dfs, axis=0)
player_df = player_stats_all.reset_index()

# Try common column names first
name_candidates = [c for c in ["player", "Player", "player_name", "name"] if c in player_df.columns]
if not name_candidates:
    # Fallback: look for anything that contains 'player' in the column name (after reset_index)
    name_candidates = [
        c for c in player_df.columns
        if "player" in str(c).lower() or str(c).lower() == "name"
    ]
if not name_candidates:
    raise KeyError(
        "Could not find a player-name column after reset_index(). "
        f"Available columns (first 30): {list(player_df.columns)[:30]}"
    )

name_col = name_candidates[0]

# Build the full unique player list (across all selected seasons)
player_list = (
    player_df[name_col]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("")]
    .drop_duplicates()
    .sort_values(kind="stable")
    .reset_index(drop=True)
    .to_frame(name="player")
)

print(f"Rows fetched: {len(player_df):,}")
print(f"Unique players: {len(player_list):,}")
print(f"Failed matches skipped: {len(failed):,}")
print("First 30 players:")
print(player_list.head(30))

# Optional: save full player list to CSV for later use
player_list.to_csv("fbref_epl_defense_players_2016-17_to_2024-25.csv", index=False)

# Optional: save failures for debugging/retry
if failed:
    pd.DataFrame(failed, columns=["match_id", "error_type", "error_message"]).to_csv(
        "fbref_failed_matches_defense_2016-17_to_2024-25.csv",
        index=False,
    )

[12/26/25 23:32:06] INFO     Saving cached data to C:\Users\LENOVO\soccerdata\data\FBref             ]8;id=606613;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=184871;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\_common.py#263\263]8;;\

                    WARNING  c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbre ]8;id=804447;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\warnings.py\warnings.py]8;;\:]8;id=564808;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\warnings.py#110\110]8;;\
                             f.py:165: FutureWarning: The behavior of DataFrame concatenation with                 
                             empty or all-NA entries is deprecated. In a future version, this will                 
                             no longer exclude empty or all-NA columns when determining the result                 
                             dtypes. To retain the old behavior, exclude the relevant entries                      
                             before the concat operation.                                                          
                               pd.concat(dfs)                                                                      
                                                                                                                   

Seasons: 2016-17 → 2024-25
Matches found in schedule: 3,420


[12/26/25 23:32:50] WARNING  c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbre ]8;id=570262;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\warnings.py\warnings.py]8;;\:]8;id=796198;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\warnings.py#110\110]8;;\
                             f.py:165: FutureWarning: The behavior of DataFrame concatenation with                 
                             empty or all-NA entries is deprecated. In a future version, this will                 
                             no longer exclude empty or all-NA columns when determining the result                 
                             dtypes. To retain the old behavior, exclude the relevant entries                      
                             before the concat operation.                                                          
                               pd.concat(dfs)                                                                      
                                                                                                                   

[12/26/25 23:32:54] INFO     [1/1] Retrieving game with id=09bf9607                                    ]8;id=559893;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=763316;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#815\815]8;;\

...

[12/26/25 23:34:04] WARNING  No stats found for home team for game with id=5e0de35a                    ]8;id=667211;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=865746;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#836\836]8;;\

                    WARNING  No stats found for away team for game with id=5e0de35a                    ]8;id=383220;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=983362;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#847\847]8;;\

                    WARNING  c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbre ]8;id=782539;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\warnings.py\warnings.py]8;;\:]8;id=822947;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\warnings.py#110\110]8;;\
                             f.py:165: FutureWarning: The behavior of DataFrame concatenation with                 
                             empty or all-NA entries is deprecated. In a future version, this will                 
                             no longer exclude empty or all-NA columns when determining the result                 
                             dtypes. To retain the old behavior, exclude the relevant entries                      
                             before the concat operation.                                                          
                               pd.concat(dfs)                                                                      
                                                                                                                   

[12/26/25 23:34:06] INFO     [1/1] Retrieving game with id=d2f7199a                                    ]8;id=204329;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=601593;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#815\815]8;;\

KeyboardInterrupt: 

In [2]:
import inspect
import pandas as pd

# See what arguments this soccerdata version expects
print("FBref.read_player_match_stats signature:")
print(inspect.signature(fbref.read_player_match_stats))

FBref.read_player_match_stats signature:
(stat_type: str = 'summary', match_id: Union[str, list[str], NoneType] = None, force_cache: bool = False) -> pandas.core.frame.DataFrame
